# Create the HARP sweep app on TAPIS (tapipy)

This notebook registers the **HARP sweep job app** on a TAPIS tenant with `tapipy`. The HARP sweep server (`harp_server/`) submits one TAPIS job of this app per chunk of the parameter sweep.

Steps:
1. Install `tapipy`
2. Log in to the TAPIS tenant
3. Choose the app id and container image
4. Create the app (or update it if it already exists)
5. Check the app
6. *(Optional)* Run one small test job on an HPC system and read its `harp_profile.csv` back through TAPIS

The container image `ghcr.io/manikyaswathi/harp-sweep-eulernumber:1.0.0` is built and pushed by the GitHub workflow `.github/workflows/build-sweep-image.yml`. Make the package **public** on GitHub once, so OSC/TACC can pull it.

Your HPC system (e.g. OSC Pitzer) and its credentials must already be registered in TAPIS — see `Executing_HARP_using_TAPIS_OSC.ipynb`.

## Step 1. Install tapipy
If `pip` fails on `atomicwrites`, uncomment the first line. On Colab, restart the runtime after installing.

In [ ]:
# %pip install atomicwrites==1.4.0
%pip install tapipy

## Step 2. Log in to TAPIS

In [ ]:
import getpass
from tapipy.tapis import Tapis

tenant_url = "https://icicle.tapis.io"   # or https://tacc.tapis.io, ...
username = input("TAPIS username: ")
password = getpass.getpass("TAPIS password: ")

client = Tapis(base_url=tenant_url, username=username, password=password)
client.get_tokens()
del password
print("Logged in as", username, "- token expires at", client.access_token.expires_at)

## Step 3. Choose the app

Set `APP` to `"euler"` or `"yolo"`. Run the notebook once per app you need.

* `app_id` — must be unique in the tenant, so it includes your username.
* `container_image` — the HARP sweep image, with `docker://` in front of the registry path.
* The resources are **defaults**; the HARP UI overrides nodes, cores, memory and minutes per system for every job.
* **YOLO on GPU:** in the HARP UI, set *Container args* to `--nv` for GPU queues so Singularity exposes the GPUs.

In [ ]:
# Which HARP app to register: "euler" or "yolo"
APP = "euler"

APPS = {
    "euler": {"image": "docker://ghcr.io/manikyaswathi/harp-sweep-eulernumber:1.0.0",
              "memoryMB": 4000, "coresPerNode": 1,
              "description": "Euler number estimation"},
    "yolo":  {"image": "docker://ghcr.io/manikyaswathi/harp-sweep-yolo:1.0.0",
              "memoryMB": 16000, "coresPerNode": 4,
              "description": "Ultralytics YOLO training on coco8 (CPU or GPU; use container args --nv on GPU queues)"},
}
preset = APPS[APP]

app_id = f"harp-sweep-{APP}-{username}"
app_version = "1.0.0"
container_image = preset["image"]
# container_image = "docker://<dockerhub-user>/harp-sweep-eulernumber:1.0.0"   # if you pushed to Docker Hub

app_def = {
    "id": app_id,
    "version": app_version,
    "description": "HARP sweep job - " + preset["description"] +
                   ". Runs a list of parameter combinations and writes harp_profile.csv to the TAPIS output directory",
    "runtime": "SINGULARITY",
    "runtimeOptions": ["SINGULARITY_RUN"],   # runs the image's ENTRYPOINT (harp_job_runner.py) with the app args
    "jobType": "BATCH",
    "containerImage": container_image,
    "jobAttributes": {
        "parameterSet": {
            "archiveFilter": {"includeLaunchFiles": False}
        },
        "nodeCount": 1,
        "coresPerNode": preset["coresPerNode"],
        "memoryMB": preset["memoryMB"],
        "maxMinutes": 60,
    },
    "tags": ["harp", "harp-sweep", APP],
}
app_def

## Step 4. Create the app — or update it if it already exists

In [ ]:
def app_exists(app_id, version):
    try:
        client.apps.getApp(appId=app_id, appVersion=version)
        return True
    except Exception:
        return False

if app_exists(app_id, app_version):
    # putApp replaces the definition of an existing version; id/version go in the URL, not the body
    body = {k: v for k, v in app_def.items() if k not in ("id", "version")}
    client.apps.putApp(appId=app_id, appVersion=app_version, **body)
    print(f"Updated app {app_id} v{app_version}")
else:
    client.apps.createAppVersion(**app_def)
    print(f"Created app {app_id} v{app_version}")

## Step 5. Check the app
The app should be listed below. It now appears in the **HARP app (container)** dropdown of the HARP UI.

In [ ]:
app = client.apps.getApp(appId=app_id, appVersion=app_version)
print("id:       ", app.id)
print("version:  ", app.version)
print("image:    ", app.containerImage)
print("runtime:  ", app.runtime, app.runtimeOptions)
print("enabled:  ", app.enabled)

print("\nAll HARP apps you can see:")
for a in client.apps.getApps(search="(id.like.harp*)", listType="ALL", limit=-1):
    print(f"  {a.id}  v{a.version}  {a.containerImage}")

## Step 6 (optional). Run one small test job

Submits **one** job with two Euler combinations to your HPC system, waits for it, and reads `harp_profile.csv` back through TAPIS Files — the same path the HARP server uses. Do this once before using the UI: it proves the image, the app, the scheduler options and the archive location all work.

Fill in your execution system, queue, allocation and a writable folder.

In [ ]:
exec_system_id = "<your-TAPIS-system-id>"          # e.g. the Pitzer system from Executing_HARP_using_TAPIS_OSC.ipynb
queue = "serial"                                    # GPU: e.g. "gpuserial-40core" on Pitzer
container_args = ""                                 # GPU: "--nv" so the container sees the GPUs
scheduler_options = "-A <Project-Account>"          # e.g. -A PAS2271
storage_system_id = exec_system_id                  # where TAPIS copies the job output
storage_dir = "/fs/scratch/<Project-Account>/<you>/harp_runs/app-test"

In [ ]:
import base64, json

TEST_JOBS = {
    "euler": {"command": "python3 calc_e.py {method} {n} {precision}", "workdir": "/app/01-eulers_number",
              "combinations": [
                  {"index": 0, "run_type": "SD", "parameters": {"method": "pow", "n": 1000, "precision": 64}},
                  {"index": 1, "run_type": "SD", "parameters": {"method": "factorial", "n": 1000, "precision": 64}}]},
    "yolo":  {"command": "python3 train_yolo.py --model {model} --epochs {epochs} --imgsz {imgsz} --batch {batch} --device {device}",
              "workdir": "/app/02-yolo",
              "combinations": [
                  {"index": 0, "run_type": "SD", "parameters": {"model": "yolo11n", "epochs": 1, "imgsz": 320, "batch": 4, "device": "auto"}},
                  {"index": 1, "run_type": "SD", "parameters": {"model": "yolo11n", "epochs": 1, "imgsz": 480, "batch": 4, "device": "auto"}}]},
}
job_spec = {"job_name": "app-test", "repetitions": 2, "run_timeout_sec": 1800, **TEST_JOBS[APP]}
spec_arg = base64.urlsafe_b64encode(json.dumps(job_spec).encode()).decode()

parameter_set = {"appArgs": [{"arg": spec_arg}],
                 "schedulerOptions": [{"arg": scheduler_options}]}
if container_args:
    parameter_set["containerArgs"] = [{"arg": container_args}]

job = client.jobs.submitJob(
    name=f"harp-{APP}-app-test",
    appId=app_id, appVersion=app_version,
    execSystemId=exec_system_id, execSystemLogicalQueue=queue,
    archiveSystemId=storage_system_id, archiveSystemDir=storage_dir, archiveOnAppError=True,
    parameterSet=parameter_set,
)
job_uuid = job.uuid
print("Submitted job", job_uuid)

In [ ]:
import time

while True:
    status = client.jobs.getJobStatus(jobUuid=job_uuid).status
    print(time.strftime("%H:%M:%S"), status)
    if status in ("FINISHED", "FAILED", "CANCELLED"):
        break
    time.sleep(30)

### Results
`tapisjob.out` is the job log. Look for the line `[HARP] writing results to ...` — it shows where the container wrote its output.

In [ ]:
log = client.jobs.getJobOutputDownload(jobUuid=job_uuid, outputPath="tapisjob.out")
print(log.decode() if isinstance(log, bytes) else log)

In [ ]:
# The profiling data, read through TAPIS Files from the archive location
csv_bytes = client.files.getContents(systemId=storage_system_id, path=f"{storage_dir}/harp_profile.csv")
print(csv_bytes.decode())

If the CSV prints with 4 rows (2 combinations × 2 repetitions), the app is ready: start the HARP server and launch sweeps from the UI.

If the job failed, check `tapisjob.out` above and the job details with `client.jobs.getJob(jobUuid=job_uuid)` (field `lastMessage`).